In [2]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Default paramaters
SEED = 460
S0 = 100
K = 95
R = 0.03
Q = 0.02
SIGMA = 0.25
M = 50
T = 0.5
NUM_PATH = 20_000

# Question 1


In [ ]:
# Question 1 functions


def GBM(
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Generate GBM with annualized paramaters
    """

    # Set seed for reproducibility
    rng = np.random.default_rng(seed=seed)
    dt = terminal_time / time_steps

    # Generate all brownian motion paths using standard normal distribution
    Z = rng.standard_normal((time_steps, num_path))

    # Compute log returns for each path and for each time step
    drift = (interest_rate - dividend_yield - 0.5 * volatility**2) * dt
    diffusion = volatility * np.sqrt(dt) * Z
    log_returns = drift + diffusion

    # Cumlate the log returns to get the log stock price
    log_prices = np.vstack([np.zeros(num_path), np.cumsum(log_returns, axis=0)])

    Ws = np.cumsum(np.vstack([np.zeros(num_path), np.sqrt(dt) * Z]), axis=0)

    # Convert log prices to actual stock prices
    stock_prices = initial_stock_price * np.exp(log_prices)

    return stock_prices, Ws


def apply_discount_factor(price, interest_rate=R, terminal_time=T):
    """
    Discount the price of an option to present value
    """
    return price * np.exp(-interest_rate * terminal_time)


def Undiscounted_Asian_Put(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Generate undiscounted payoff for Asian put option where stock prices are modeled using GBM
    """
    # Compute the stock price paths using GBM
    stock_prices, _ = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the payoff of the Asian put option for each path
    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    return payoffs


def Monte_Carlo_Simulation(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )
    put_prices = apply_discount_factor(put_payoffs, interest_rate, terminal_time)

    return put_prices, np.mean(put_prices)

In [7]:
# Question 1 example usage
put_prices, average_put_price = Monte_Carlo_Simulation()
print(f"Average Asian Put Price: {average_put_price:.4f}")

Average Asian Put Price: 1.8680


# Question 5


In [ ]:
# Question 5 functions

# Finite difference step size 1 dollar in initial stock price
STOCK_PRICE_DELTA = 1

# Finite difference step size 1%
VOLATILITY_DELTA = 0.01


# Finite Difference Methods
def finite_difference_method_delta(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    h=STOCK_PRICE_DELTA,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Compute the delta of an Asian put option using finite difference method
    """

    # Compute the price of the Asian put option at S0 + h
    plus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price + h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    mean_plus_h_payoffs = np.mean(plus_h_put_payoffs)

    # Compute the price of the Asian put option at S0 - h
    minus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price - h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )
    mean_minus_h_payoffs = np.mean(minus_h_put_payoffs)

    # Compute the delta using finite difference method
    delta = apply_discount_factor(
        (mean_plus_h_payoffs - mean_minus_h_payoffs) / (2 * h),
        interest_rate,
        terminal_time,
    )

    return delta


def finite_difference_method_vega(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    h=VOLATILITY_DELTA,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Compute the vega of an Asian put option using finite difference method
    """

    # Compute the price of the Asian put option at S0 with volatility + h
    plus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility + h,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    mean_plus_h_payoffs = np.mean(plus_h_put_payoffs)

    # Compute the price of the Asian put option at S0 with volatility - h
    minus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility - h,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    mean_minus_h_payoffs = np.mean(minus_h_put_payoffs)

    # Compute the vega using finite difference method
    vega = apply_discount_factor(
        (mean_plus_h_payoffs - mean_minus_h_payoffs) / (2 * h),
        interest_rate,
        terminal_time,
    )

    return vega


def finite_difference_method_gamma(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    h=STOCK_PRICE_DELTA,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Compute the gamma of an Asian put option using finite difference method
    """

    # Compute the price of the Asian put option at S0 + h
    plus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price + h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )
    mean_plus_h_payoffs = np.mean(plus_h_put_payoffs)

    # Compute the price of the Asian put option at S0 - h
    minus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price - h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    mean_minus_h_payoffs = np.mean(minus_h_put_payoffs)

    # Compute the price of the Asian put option at S0
    put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )
    mean_put_payoffs = np.mean(put_payoffs)

    # Compute the gamma using finite difference method
    gamma = apply_discount_factor(
        (mean_plus_h_payoffs - 2 * mean_put_payoffs + mean_minus_h_payoffs) / (h**2),
        interest_rate,
        terminal_time,
    )

    return gamma


# Pathwise Estimators
def pathwise_estimator_delta(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Stock price at time T
    stock_prices, _ = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    indicators = (np.mean(stock_prices[1:, :], axis=0) < strike_price).astype(float)

    return np.mean(
        (-1)
        * indicators
        * np.mean(stock_prices[1:, :], axis=0)
        / initial_stock_price
        * np.exp(-interest_rate * terminal_time)
    )


def pathwise_estimator_vega(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Stock price at time T
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    indicators = (np.mean(stock_prices[1:, :], axis=0) < strike_price).astype(float)

    times = np.linspace(0, terminal_time, time_steps + 1)[:, None]

    return apply_discount_factor(
        np.mean(
            -indicators
            * np.mean(
                stock_prices[1:, :] * (Ws[1:, :] - volatility * times[1:, :]), axis=0
            )
        ),
        interest_rate,
        terminal_time,
    )


def likelihood_estimator_delta(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Stock price at time T
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    dt = terminal_time / time_steps

    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    return apply_discount_factor(
        np.mean(payoffs * Ws[1, :] / (initial_stock_price * volatility * dt)),
        interest_rate,
        terminal_time,
    )


def likelihood_estimator_vega(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Stock price at time T
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    dt = terminal_time / time_steps

    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    dW = np.diff(Ws, axis=0)

    return apply_discount_factor(
        np.mean(
            payoffs
            * (
                -time_steps / volatility
                + np.sum(dW**2, axis=0) / (volatility * dt)
                - Ws[-1, :]
            )
        ),
        interest_rate,
        terminal_time,
    )


def likelihood_estimator_gamma(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Stock price at time T
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    dt = terminal_time / time_steps

    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    L_s0 = Ws[1, :] / (initial_stock_price * volatility * dt)

    return apply_discount_factor(
        np.mean(
            payoffs
            * (
                L_s0**2
                - L_s0 / initial_stock_price
                - 1 / (initial_stock_price**2 * volatility**2 * dt)
            )
        ),
        interest_rate,
        terminal_time,
    )